# Skewness-Align: end-to-end latency and accuracy

**Run All on classic dedicated compute (tested locally with Spark 3.5 / Scala 2.12; the repository uses DBR 14.3).**
The existing JAR and `range_query` inputs must be readable in the configured Unity Catalog Volumes.
The notebook loads the JAR itself; no new Java build, predictor branch, or model is required.
It calls the original `RangeQueryWithRatio` and beta implementation without modifying them.
Routing to exact execution is an experiment policy implemented here, not a pre-existing service in `main`.

Defaults are a **pilot**: 100 calibration queries, 100 held-out queries, three timed repetitions,
one merge rate near 0.5, and two synopsis-access modes. Increase `test_queries` (e.g. 1000),
`repetitions` (e.g. 5), and `merge_rates` (e.g. `1,0.5,0.1`) for a larger experiment.
Every fallback and always-exact query really executes Spark `COUNT`, using the same data and predicates.
This can take several minutes or longer; the notebook prints progress and saves each completed pair.

* `resident`: synopsis and beta are prepared once; their setup cost is reported separately.
* `reload`: each approximate request includes reading/deserializing its synopsis and preparing beta.
Neither mode claims cold physical storage: Spark, Databricks disk, and OS caches may already be warm.
No raw DataFrame is explicitly cached. JVM/Spark warm-up is excluded from measured repetitions.

The default exact representation is a run-specific Parquet copy of the CSV points, prepared offline.
Choose `raw_format=csv` to query the original tab-separated files instead. Both methods always use
the same representation. Synopsis files retain the repository's existing text format.
All outputs go to a new run directory; existing inputs and previous results are never overwritten.

In [0]:
import java.io.{File, PrintWriter}
import java.net.URLClassLoader
import java.nio.charset.StandardCharsets.UTF_8
import java.nio.file.{Files, Paths}
import java.security.MessageDigest
import java.util.UUID
import scala.collection.mutable.ArrayBuffer
import scala.io.Source
import scala.util.Random
import scala.util.control.NonFatal
import org.apache.spark.sql.{DataFrame, Row}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

val defaultBase = "/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake"
dbutils.widgets.text("volume_base", defaultBase)
dbutils.widgets.text("jar_path", "")
dbutils.widgets.text("data_dir", "")
dbutils.widgets.text("query_file", "")
dbutils.widgets.text("output_root", "")
dbutils.widgets.text("calibration_queries", "100")
dbutils.widgets.text("test_queries", "100")
dbutils.widgets.text("repetitions", "3")
dbutils.widgets.text("warmup_queries", "5")
dbutils.widgets.text("merge_rates", "0.5")
dbutils.widgets.text("epsilon", "0.10")
dbutils.widgets.text("delta", "0.05")
dbutils.widgets.text("seed", "0")
dbutils.widgets.dropdown("synopsis_access", "both", Seq("both", "resident", "reload"))
dbutils.widgets.dropdown("raw_format", "parquet", Seq("parquet", "csv"))
dbutils.widgets.dropdown("reproduce_authors", "true", Seq("true", "false"))
dbutils.widgets.text("logger_heap", "4g")

def setting(name: String): String = dbutils.widgets.get(name).trim
def orDefault(name: String, fallback: String): String = Option(setting(name)).filter(_.nonEmpty).getOrElse(fallback)
val base = setting("volume_base").stripSuffix("/")
val jarPath = orDefault("jar_path", s"$base/jars/synopsislake.jar")
val dataDir = orDefault("data_dir", s"$base/data/range_query").stripSuffix("/")
val queryPath = orDefault("query_file", s"$dataDir/dataDrivenQuery/srcQuery-40-56-32-48-Ans.csv")
val outputRoot = orDefault("output_root", s"$base/output/e2e_skewness_align").stripSuffix("/")
val nCalibration = setting("calibration_queries").toInt
val nTest = setting("test_queries").toInt
val repetitions = setting("repetitions").toInt
val warmupQueries = setting("warmup_queries").toInt
val epsilon = setting("epsilon").toDouble
val delta = setting("delta").toDouble
val seed = setting("seed").toLong
val requestedRates = setting("merge_rates").split(",").map(_.trim.toDouble).distinct
val accessModes = if (setting("synopsis_access") == "both") Seq("resident", "reload") else Seq(setting("synopsis_access"))
val rawFormat = setting("raw_format")
val reproduceAuthors = setting("reproduce_authors").toBoolean
require(nCalibration >= 10 && nTest >= 10, "Use at least 10 calibration and 10 test queries.")
require(repetitions >= 1 && warmupQueries >= 0)
require(epsilon > 0 && epsilon < 1 && delta >= 0 && delta < 1)
require(requestedRates.nonEmpty && requestedRates.forall(r => r > 0 && r <= 1))
require(setting("logger_heap").matches("[1-9][0-9]*[mMgG]"), "logger_heap must be a Java heap size, e.g. 4g")
val rawPaths = (0 until 4).map(i => s"$dataDir/$i.csv")
val requiredPaths = Seq(jarPath, queryPath) ++ rawPaths
requiredPaths.foreach(p => require(Files.isReadable(Paths.get(p)) && Files.size(Paths.get(p)) > 0,
  s"Missing/unreadable/empty: $p. Use dedicated compute with Volume access; the existing CI uploads these inputs."))
val runId = java.time.Instant.now().toString.replace(":", "-") + "_" + UUID.randomUUID().toString.take(8)
val runDir = s"$outputRoot/$runId"
Files.createDirectories(Paths.get(runDir))
// Shared with the Python report cell; this is a path, not a persistent table.
spark.conf.set("synopsislake.e2e.runDir", runDir)
println(s"Results: $runDir")

## Helpers, original-JAR access, and input validation
A disjoint 50/20/30 split reserves the first 50% for future training, calibrates on 20%, and tests on 30%.
Identical query rectangles are deduplicated before splitting to prevent duplicate-query leakage.
No small-count or zero-count queries are filtered out. Requested sample sizes are capped by split size.

In [0]:
object E2E {
  def finite(x: Double): Boolean = !x.isNaN && !x.isInfinity
  def lines(path: String): Vector[String] = {
    val src = Source.fromFile(path, "UTF-8")
    try src.getLines().toVector finally src.close()
  }
  def sha256(path: String): String = {
    val digest = MessageDigest.getInstance("SHA-256")
    val in = Files.newInputStream(Paths.get(path))
    val buffer = new Array[Byte](1024 * 1024)
    try { var n = in.read(buffer); while (n != -1) { digest.update(buffer, 0, n); n = in.read(buffer) } }
    finally in.close()
    digest.digest().map(b => f"${b & 0xff}%02x").mkString
  }
  def csv(values: Seq[Any]): String = values.map { x =>
    val s = Option(x).map(_.toString).getOrElse("")
    "\"" + s.replace("\"", "\"\"") + "\""
  }.mkString(",")
  def writeCsv(path: String, header: Seq[String], rows: Iterable[Seq[Any]]): Unit = {
    val out = new PrintWriter(Files.newBufferedWriter(Paths.get(path), UTF_8))
    try { out.println(csv(header)); rows.foreach(r => out.println(csv(r))); require(!out.checkError(), s"Write failed: $path") }
    finally out.close()
  }
  def ms(start: Long): Double = (System.nanoTime() - start) / 1e6
  case class Query(id: Int, minLon: Double, maxLon: Double, minLat: Double, maxLat: Double, suppliedExact: Double) {
    def bounds: Vector[Double] = Vector(minLon, maxLon, minLat, maxLat)
    def loggerLine: String = (bounds :+ suppliedExact).mkString(",")
  }
  case class Loaded(hist: AnyRef, beta: AnyRef, rate: Double, loadMs: Double, betaMs: Double)
  case class Score(estimate: Double, score: Double, overlap: Double, cells: Double)
  // Zero-count policy: only an exactly zero estimate is acceptable. Relative error remains undefined.
  def safe(estimate: Double, exact: Double, eps: Double): Boolean =
    finite(estimate) && (if (exact == 0) estimate == 0 else math.abs(estimate - exact) / exact <= eps)
  def calibrate(rows: Seq[(Double, Boolean)], maxViolation: Double): Double = {
    val sorted = rows.filter(r => finite(r._1)).sortBy(_._1)
    var bad = 0
    var chosen = Double.NegativeInfinity
    sorted.indices.foreach { i =>
      if (!sorted(i)._2) bad += 1
      val tieEnd = i == sorted.size - 1 || sorted(i + 1)._1 != sorted(i)._1
      if (tieEnd && bad.toDouble / (i + 1) <= maxViolation) chosen = sorted(i)._1
    }
    chosen
  }
  // Reflection only loads/calls the existing Java implementation; no histogram algorithm is reimplemented.
  class OriginalJar(path: String) {
    val loader = new URLClassLoader(Array(new File(path).toURI.toURL), Thread.currentThread().getContextClassLoader)
    val histClass = loader.loadClass("histogram.SimpleSpatialHistogramOpt")
    val betaClass = loader.loadClass("org.apache.commons.math3.distribution.BetaDistribution")
    val helperClass = loader.loadClass("utils.UtilsFunctionHistogram")
    loader.loadClass("sigspatial25.ErrorPredictionLogExp")
    val loadedJar = new File(histClass.getProtectionDomain.getCodeSource.getLocation.toURI).getPath
    require(sha256(loadedJar) == sha256(path),
      s"A different SynopsisLake JAR is already loaded: $loadedJar. Detach it/restart compute or select that JAR explicitly.")
    val mathJar = new File(betaClass.getProtectionDomain.getCodeSource.getLocation.toURI).getPath
    private val d = java.lang.Double.TYPE
    private val i = java.lang.Integer.TYPE
    private val ctor = histClass.getConstructor(d, d, d, d, i, i, classOf[Array[Array[Double]]], classOf[Array[Double]], classOf[Array[Double]])
    private val betaMethod = helperClass.getMethod("ComputeBetaDist", classOf[Array[Array[Double]]], classOf[Array[Double]], classOf[Array[Double]], d, d)
    private val queryMethod = histClass.getMethod("RangeQueryWithRatio", betaClass, d, d, d, d)
    def load(file: String): Loaded = {
      val t0 = System.nanoTime()
      val text = lines(file)
      require(text.size >= 4, s"Invalid histogram: $file")
      val rate = text.head.toDouble
      val lon = text(1).split(",").map(_.toDouble)
      val lat = text(2).split(",").map(_.toDouble)
      val counts = text.drop(3).map(_.split(",").map(_.toDouble)).toArray
      require(lon.length >= 2 && lat.length >= 2 && counts.length == lon.length - 1 && counts.forall(_.length == lat.length - 1))
      require(lon.sliding(2).forall(a => finite(a(0)) && finite(a(1)) && a(1) > a(0)))
      require(lat.sliding(2).forall(a => finite(a(0)) && finite(a(1)) && a(1) > a(0)))
      require(counts.flatten.forall(x => finite(x) && x >= 0))
      val hist = ctor.newInstance(Double.box(lon.head), Double.box(lat.head), Double.box(lon.last), Double.box(lat.last),
        Int.box(lon.length - 1), Int.box(lat.length - 1), counts, lon, lat).asInstanceOf[AnyRef]
      val loadMs = ms(t0)
      val t1 = System.nanoTime()
      // Preserve main's original stride and constants; fixStride=false in the logger too.
      val beta = betaMethod.invoke(null, counts, lon, lat, Double.box(0.5), Double.box(1e-6))
      Loaded(hist, beta, rate, loadMs, ms(t1))
    }
    def score(h: Loaded, q: Query): Score = {
      val a = queryMethod.invoke(h.hist, h.beta, Double.box(q.minLon), Double.box(q.maxLon),
        Double.box(q.minLat), Double.box(q.maxLat)).asInstanceOf[Array[Double]]
      Score(a(0), a(2), if (a.length > 3) a(3) else 0.0, if (a.length > 4) a(4) else 0.0)
    }
  }
}
import E2E._
val setupRows = ArrayBuffer.empty[Seq[Any]]
val jarStart = System.nanoTime()
val api = new OriginalJar(jarPath)
setupRows += Seq("jar_load_and_identity_check", ms(jarStart), "driver", "")
val inputQueries = lines(queryPath).zipWithIndex.map { case (line, id) =>
  val v = line.split(",", -1).map(_.trim.toDouble)
  require(v.length == 5 && v.forall(finite), s"Query line ${id + 1} needs five finite numbers.")
  require(v(0) < v(1) && v(2) < v(3) && v(4) >= 0 && v(4) == math.rint(v(4)), s"Invalid query line ${id + 1}")
  Query(id, v(0), v(1), v(2), v(3), v(4))
}
val queryGroups = inputQueries.groupBy(_.bounds)
require(queryGroups.values.forall(g => g.map(_.suppliedExact).distinct.size == 1), "Duplicate rectangles have conflicting ground truths.")
val distinctQueries = queryGroups.values.map(_.minBy(_.id)).toVector.sortBy(_.id)
val shuffled = new Random(seed).shuffle(distinctQueries)
val calibrationQueries = shuffled.slice(shuffled.size / 2, (shuffled.size * 0.7).toInt).take(nCalibration)
val testQueries = shuffled.drop((shuffled.size * 0.7).toInt).take(nTest)
require(calibrationQueries.size >= 10 && testQueries.size >= 10, "Not enough distinct queries for calibration/test splits.")
val selectedQueries = calibrationQueries ++ testQueries
val loggerIndex = selectedQueries.zipWithIndex.map { case (q, i) => q.id -> i }.toMap
writeCsv(s"$runDir/queries.csv", Seq("qId", "split", "minLon", "maxLon", "minLat", "maxLat", "supplied_exact"),
  selectedQueries.map(q => Seq(q.id, if (calibrationQueries.contains(q)) "calibration" else "test") ++ q.bounds ++ Seq(q.suppliedExact)))
println(s"Selected ${calibrationQueries.size} calibration and ${testQueries.size} test queries from ${distinctQueries.size} distinct rectangles.")
println(s"Removed ${inputQueries.size - distinctQueries.size} duplicate rectangles; no count-based filtering.")

## Prepare reusable synopses with the existing logger
The cache key includes JAR/query hashes, input paths/sizes/modification times, and construction settings.
Raw files are assumed immutable for this run. `reproduce_authors=true` matches notebook 01b.
This is **offline pipeline time**, including local staging and the logger's evaluation/export work,
not pure construction time. Python stages the Java subprocess's inputs on the driver because
Databricks does not support Volume access from subprocesses launched by Scala.

In [0]:
val rawIdentity = rawPaths.map(p => s"$p|${Files.size(Paths.get(p))}|${Files.getLastModifiedTime(Paths.get(p)).toMillis}").mkString("\n")
val loggerQueryText = selectedQueries.map(_.loggerLine).mkString("", "\n", "\n")
val buildIdentity = s"${sha256(jarPath)}\n$rawIdentity\n$loggerQueryText\n40,56,32,48|42|8|$reproduceAuthors|fixStride=false"
val artifactKey = MessageDigest.getInstance("SHA-256").digest(buildIdentity.getBytes(UTF_8)).map(b => f"${b & 0xff}%02x").mkString
val artifactDir = s"$outputRoot/artifacts/$artifactKey"
val completeMarker = Paths.get(artifactDir, "COMPLETE")
val artifactStart = System.nanoTime()
val reusedArtifacts = Files.isRegularFile(completeMarker) && new String(Files.readAllBytes(completeMarker), UTF_8) == buildIdentity
if (!reusedArtifacts) {
  Files.createDirectories(Paths.get(artifactDir))
  val buildQueryFile = s"$artifactDir/logger_queries.csv"
  Files.write(Paths.get(buildQueryFile), loggerQueryText.getBytes(UTF_8))
}
val buildPlan = new java.util.TreeMap[String, String]()
Seq("reuse" -> reusedArtifacts.toString, "jar" -> jarPath, "math_jar" -> api.mathJar,
  "java" -> s"${System.getProperty("java.home")}/bin/java", "heap" -> setting("logger_heap"),
  "data_dir" -> dataDir, "query_file" -> s"$artifactDir/logger_queries.csv", "artifact_dir" -> artifactDir,
  "log_file" -> s"$runDir/logger.log", "reproduce_authors" -> reproduceAuthors.toString
).foreach { case (k, v) => buildPlan.put(k, v) }
spark.conf.set("synopsislake.e2e.buildPlan", new com.fasterxml.jackson.databind.ObjectMapper().writeValueAsString(buildPlan))

In [0]:
%python
import json
import os
from pathlib import Path
import shutil
import subprocess
import tempfile

build_plan = json.loads(spark.conf.get("synopsislake.e2e.buildPlan"))
if build_plan["reuse"] != "true":
    local_parent = "/local_disk0/tmp" if Path("/local_disk0/tmp").is_dir() else None
    with tempfile.TemporaryDirectory(prefix="synopsislake_build_", dir=local_parent) as scratch:
        scratch = Path(scratch)
        local_data, local_output = scratch / "data", scratch / "output"
        local_data.mkdir()
        local_output.mkdir()
        print("Staging inputs locally for the unchanged Java logger...")
        for index in range(4):
            shutil.copyfile(Path(build_plan["data_dir"]) / f"{index}.csv", local_data / f"{index}.csv")
        local_jar, local_math, local_queries = scratch / "synopsislake.jar", scratch / "math.jar", scratch / "queries.csv"
        for source, target in [(build_plan["jar"], local_jar), (build_plan["math_jar"], local_math),
                               (build_plan["query_file"], local_queries)]:
            shutil.copyfile(source, target)
        command = [build_plan["java"], f"-Xmx{build_plan['heap']}", "-cp", f"{local_jar}{os.pathsep}{local_math}",
                   "sigspatial25.ErrorPredictionLogExp", "-dataDir", str(local_data), "-numFile", "4",
                   "-srcRes", "40,56,32,48", "-queryFile", str(local_queries), "-outDir", str(local_output),
                   "-end", "42", "-step", "8", "-reproduceAuthors", build_plan["reproduce_authors"], "-fixStride", "false"]
        local_log = scratch / "logger.log"
        try:
            with local_log.open("w") as log:
                subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, check=True, timeout=3600)
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
            tail = "\n".join(local_log.read_text(errors="replace").splitlines()[-20:])
            raise RuntimeError(f"Logger failed; see {build_plan['log_file']}\n{tail}") from exc
        finally:
            if local_log.exists():
                shutil.copyfile(local_log, build_plan["log_file"])
        for artifact in local_output.iterdir():
            shutil.copyfile(artifact, Path(build_plan["artifact_dir"]) / artifact.name)
        print(f"Synopses exported to {build_plan['artifact_dir']}")
else:
    print("Reusing previously validated synopsis artifacts.")

In [0]:
if (!reusedArtifacts) {
  require((0 until 4).forall(i => Files.isRegularFile(Paths.get(artifactDir, s"src_$i.txt"))))
  require(new File(artifactDir).listFiles().count(_.getName.matches("arm_\\d+\\.txt")) == 18,
    "Expected 18 merged histograms for the configured source resolutions; incomplete or incompatible logger output.")
  require(lines(s"$artifactDir/rows.csv").size == selectedQueries.size * 18 + 1, "Logger did not produce every expected query/rate row.")
  Files.write(completeMarker, buildIdentity.getBytes(UTF_8))
}
setupRows += Seq(if (reusedArtifacts) "artifact_cache_validation" else "offline_logger_pipeline", ms(artifactStart), "offline", artifactDir)
val artifactFiles = new File(artifactDir).listFiles().filter(_.getName.matches("arm_\\d+\\.txt")).toSeq
val availableRates = artifactFiles.map(f => (lines(f.getPath).head.toDouble, f.getPath)).sortBy(-_._1)
val selectedRates = requestedRates.map(r => availableRates.minBy(x => math.abs(x._1 - r))).distinct.toSeq
require(selectedRates.nonEmpty)
println(s"Requested rates: ${requestedRates.mkString(", ")}; actual available rates selected: ${selectedRates.map(_._1).mkString(", ")}")
val residentHistograms = selectedRates.map { case (rate, file) =>
  val h = api.load(file)
  setupRows += Seq("synopsis_load", h.loadMs, rate.toString, file)
  setupRows += Seq("beta_preparation", h.betaMs, rate.toString, "")
  rate -> h
}.toMap
// Verify serialization and query results against the untouched logger, including the score.
val loggerRows = lines(s"$artifactDir/rows.csv").tail.map(_.split(",")).map(a => (a(1).toDouble, a(4).toInt) -> a).toMap
def closeEnough(x: Double, y: Double): Boolean = (x.isNaN && y.isNaN) || x == y || math.abs(x - y) <= 1e-8 * math.max(1.0, math.abs(y))
for ((rate, _) <- selectedRates; q <- selectedQueries) {
  val result = api.score(residentHistograms(rate), q)
  val logged = loggerRows((rate, loggerIndex(q.id)))
  require(closeEnough(result.estimate, logged(6).toDouble) && closeEnough(result.score, logged(8).toDouble),
    s"Saved histogram differs from original logger at rate=$rate qId=${q.id}")
}
println("Original logger / loaded histogram parity: PASS")

## Exact executor and calibration
Exact queries count every point in all four files, with inclusive bounds on both axes and no deduplication.
Calibration ground truths are recomputed from raw data before selecting any cutoff.
Every timed always-exact result also checks the supplied test ground truth.
Cutoffs are chosen per merge rate, keeping score ties together. They target empirical calibration
violation `delta`; this is not a confidence guarantee. Nonfinite scores always fall back.

In [0]:
val exactSetupStart = System.nanoTime()
// The authors' files also contain timestamps. Like the Java loader, read the first two tab-separated
// fields and ignore trailing fields; a two-column FAILFAST CSV schema would reject these valid rows.
val rawFields = split(col("value"), "\t")
val inputPoints = spark.read.text(rawPaths: _*).select(
  rawFields.getItem(0).cast(DoubleType).as("lon"), rawFields.getItem(1).cast(DoubleType).as("lat"))
val rawStats = inputPoints.agg(count(lit(1)).as("n"),
  sum(when(col("lon").isNull || col("lat").isNull || isnan(col("lon")) || isnan(col("lat")) ||
    abs(col("lon")) === lit(Double.PositiveInfinity) || abs(col("lat")) === lit(Double.PositiveInfinity), 1).otherwise(0)).as("invalid")).first()
val rawCount = rawStats.getLong(0)
require(rawCount > 0 && rawStats.getLong(1) == 0, "Raw data is empty or contains invalid coordinates.")
val exactDataPath = if (rawFormat == "parquet") s"$runDir/raw_points.parquet" else dataDir
if (rawFormat == "parquet") inputPoints.write.mode("errorifexists").parquet(exactDataPath)
val points = if (rawFormat == "parquet") spark.read.parquet(exactDataPath) else inputPoints
require(points.count() == rawCount, "Exact snapshot row count changed.")
setupRows += Seq("raw_validation_and_exact_representation", ms(exactSetupStart), "offline", exactDataPath)
def exactCount(q: Query): Double = {
  // Construct + execute the same plan in both arms. count() is the blocking Spark action.
  points.filter(col("lon") >= lit(q.minLon) && col("lon") <= lit(q.maxLon) &&
    col("lat") >= lit(q.minLat) && col("lat") <= lit(q.maxLat)).count().toDouble
}
val calibrationStart = System.nanoTime()
val calibrationTruth = calibrationQueries.zipWithIndex.map { case (q, index) =>
  val exact = exactCount(q)
  require(exact == q.suppliedExact, s"Calibration ground truth mismatch qId=${q.id}: file=${q.suppliedExact}, Spark=$exact. Check dataset/query pairing.")
  if ((index + 1) % 25 == 0) println(s"Verified calibration ${index + 1}/${calibrationQueries.size}")
  q.id -> exact
}.toMap
val cutoffs = selectedRates.map { case (rate, _) =>
  val pairs = calibrationQueries.map { q =>
    val r = api.score(residentHistograms(rate), q)
    (r.score, safe(r.estimate, calibrationTruth(q.id), epsilon))
  }
  rate -> calibrate(pairs, delta)
}.toMap
val calibrationRows = selectedRates.flatMap { case (rate, _) => calibrationQueries.map { q =>
  val s = api.score(residentHistograms(rate), q)
  Seq(rate, q.id, calibrationTruth(q.id), s.estimate, s.score, cutoffs(rate), finite(s.score) && s.score <= cutoffs(rate), safe(s.estimate, calibrationTruth(q.id), epsilon))
}}
writeCsv(s"$runDir/calibration.csv", Seq("merge_rate", "qId", "exact", "synopsis_estimate", "score", "cutoff", "accepted", "safe"), calibrationRows)
setupRows += Seq("exact_calibration_and_cutoff_selection", ms(calibrationStart), "offline", "")
cutoffs.toSeq.sortBy(-_._1).foreach { case (rate, cutoff) => println(s"rate=$rate cutoff=$cutoff") }
writeCsv(s"$runDir/setup.csv", Seq("operation", "elapsed_ms", "scope", "path"), setupRows)

## Timed paired experiment
The outer timer covers request dispatch, optional synopsis I/O + beta, the original combined score/answer
computation, routing, and any exact Spark fallback. Feature extraction and score/answer calculation are
combined in the original method; they cannot be separated honestly without instrumenting that code.
There is no model inference in this baseline. Result serialization, plots, and ground-truth comparisons
happen after the timer stops. Exact execution never pays synopsis/gate overhead.

Both members of each pair run, in randomized order; every repetition shuffles queries and configurations.
Warm-ups use calibration queries. Access modes describe synopsis residency, not raw-storage cache temperature.
Initialization and offline preparation are in `setup.csv`; they are not hidden in query latency.

In [0]:
val measurementHeader = Seq("phase", "repetition", "qId", "merge_rate", "access_mode", "method", "pair_position",
  "answer", "synopsis_estimate", "score", "cutoff", "accepted", "fallback", "total_ms", "synopsis_load_ms",
  "beta_ms", "score_and_answer_ms", "routing_ms", "exact_ms", "cells_touched", "overlap_sum", "status", "error")
case class Measurement(answer: Double, estimate: Double, score: Double, accepted: Boolean, fallback: Boolean,
  totalMs: Double, loadMs: Double, betaMs: Double, scoreMs: Double, routingMs: Double, exactMs: Double,
  cells: Double, overlap: Double, status: String, error: String)
def request(q: Query, rate: Double, file: String, mode: String, method: String): Measurement = {
  var answer = Double.NaN; var estimate = Double.NaN; var score = Double.NaN
  var accepted = false; var fallback = false
  var loadMs = 0.0; var betaMs = 0.0; var scoreMs = 0.0; var routingMs = 0.0; var exactMs = 0.0
  var cells = Double.NaN; var overlap = Double.NaN
  val start = System.nanoTime()
  try {
    if (method == "always_exact") {
      val t = System.nanoTime(); answer = exactCount(q); exactMs = ms(t)
    } else {
      val h = if (mode == "reload") api.load(file) else residentHistograms(rate)
      if (mode == "reload") { loadMs = h.loadMs; betaMs = h.betaMs }
      val tScore = System.nanoTime()
      val s = api.score(h, q)
      scoreMs = ms(tScore); estimate = s.estimate; score = s.score; cells = s.cells; overlap = s.overlap
      val tRoute = System.nanoTime()
      accepted = finite(score) && finite(estimate) && score <= cutoffs(rate)
      fallback = !accepted
      routingMs = ms(tRoute)
      if (accepted) answer = estimate
      else { val t = System.nanoTime(); answer = exactCount(q); exactMs = ms(t) }
    }
    val totalMs = ms(start)
    require(finite(answer), "Nonfinite returned answer")
    require(method != "always_exact" && !fallback || answer == q.suppliedExact,
      s"Exact count mismatch: supplied=${q.suppliedExact}, observed=$answer")
    Measurement(answer, estimate, score, accepted, fallback, totalMs, loadMs, betaMs, scoreMs, routingMs, exactMs, cells, overlap, "ok", "")
  } catch {
    case NonFatal(e) => Measurement(answer, estimate, score, accepted, fallback, ms(start), loadMs, betaMs, scoreMs,
      routingMs, exactMs, cells, overlap, "failed", Option(e.getMessage).getOrElse(e.toString))
  }
}
val configurations = for ((rate, file) <- selectedRates; mode <- accessModes) yield (rate, file, mode)
val measurementsPath = s"$runDir/measurements.csv"
val measurementOut = new PrintWriter(Files.newBufferedWriter(Paths.get(measurementsPath), UTF_8))
measurementOut.println(csv(measurementHeader))
var failedRequests = 0
var measuredRequests = 0
val benchmarkStart = System.nanoTime()
try {
  for ((rate, file, mode) <- configurations; q <- calibrationQueries.take(warmupQueries); method <- Seq("always_exact", "skewness_align")) {
    val r = request(q, rate, file, mode, method)
    require(r.status == "ok", s"Warm-up failed: ${r.error}")
  }
  for (rep <- 0 until repetitions) {
    val rng = new Random(seed + 1009L * (rep + 1))
    val work = rng.shuffle(for (q <- testQueries; config <- configurations) yield (q, config))
    work.zipWithIndex.foreach { case ((q, (rate, file, mode)), pairIndex) =>
      rng.shuffle(Seq("always_exact", "skewness_align")).zipWithIndex.foreach { case (method, position) =>
        val r = request(q, rate, file, mode, method)
        measuredRequests += 1
        if (r.status != "ok") failedRequests += 1
        measurementOut.println(csv(Seq("measured", rep, q.id, rate, mode, method, position, r.answer, r.estimate,
          r.score, cutoffs(rate), r.accepted, r.fallback, r.totalMs, r.loadMs, r.betaMs, r.scoreMs, r.routingMs,
          r.exactMs, r.cells, r.overlap, r.status, r.error)))
      }
      measurementOut.flush()
      require(!measurementOut.checkError(), s"Cannot save results to $measurementsPath")
      if ((pairIndex + 1) % 25 == 0 || pairIndex == work.size - 1)
        println(s"Repetition ${rep + 1}/$repetitions: ${pairIndex + 1}/${work.size} pairs; failures=$failedRequests")
    }
  }
} finally { measurementOut.close() }
val benchmarkWallMs = ms(benchmarkStart)
println(s"Completed $measuredRequests requests; $failedRequests failed. Saved $measurementsPath")

## Run manifest and reproducibility

In [0]:
val manifest = new java.util.TreeMap[String, String]()
Seq(
  "run_id" -> runId, "run_dir" -> runDir, "notebook_version" -> "2",
  "source_baseline" -> "main a42efb6a10582c80e1476390825b9c268856cf0e",
  "jar_path" -> jarPath, "jar_sha256" -> sha256(jarPath), "loaded_jar" -> api.loadedJar,
  "query_path" -> queryPath, "query_sha256" -> sha256(queryPath), "raw_file_identity" -> rawIdentity,
  "raw_count" -> rawCount.toString, "raw_format" -> rawFormat, "exact_data_path" -> exactDataPath,
  "artifact_dir" -> artifactDir, "artifact_cache_reused" -> reusedArtifacts.toString,
  "reproduce_authors" -> reproduceAuthors.toString, "fix_stride" -> "false",
  "epsilon" -> epsilon.toString, "delta" -> delta.toString, "seed" -> seed.toString,
  "calibration_queries" -> calibrationQueries.size.toString, "test_queries" -> testQueries.size.toString,
  "repetitions" -> repetitions.toString, "warmup_queries" -> warmupQueries.toString,
  "requested_merge_rates" -> requestedRates.mkString(","), "actual_merge_rates" -> selectedRates.map(_._1).mkString(","),
  "synopsis_access" -> accessModes.mkString(","), "raw_cache_policy" -> "no explicit DataFrame cache; OS/Databricks caches uncontrolled",
  "spark_version" -> spark.version, "scala_version" -> scala.util.Properties.versionNumberString,
  "java_version" -> System.getProperty("java.version"), "driver_max_heap_bytes" -> Runtime.getRuntime.maxMemory().toString,
  "benchmark_wall_ms_including_warmup" -> benchmarkWallMs.toString,
  "failed_requests" -> failedRequests.toString,
  "synopsis_bytes_selected" -> selectedRates.map(x => Files.size(Paths.get(x._2))).sum.toString,
  "raw_csv_bytes" -> rawPaths.map(p => Files.size(Paths.get(p))).sum.toString
).foreach { case (k, v) => manifest.put(k, v) }
Seq("spark.databricks.clusterUsageTags.sparkVersion", "spark.databricks.clusterUsageTags.clusterId",
  "spark.databricks.clusterUsageTags.clusterNodeType", "spark.databricks.clusterUsageTags.clusterWorkers",
  "spark.databricks.io.cache.enabled", "spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled").foreach { key =>
  manifest.put(key, spark.conf.getOption(key).getOrElse("unavailable"))
}
val mapper = new com.fasterxml.jackson.databind.ObjectMapper()
Files.write(Paths.get(runDir, "manifest.json"), mapper.writerWithDefaultPrettyPrinter().writeValueAsBytes(manifest))
// Safe cross-language transfer: the report reads only this run's files.
println(s"Report input ready: $runDir")

## Metrics and plots
Accuracy is evaluated once per distinct query/configuration, not once per timing repetition.
Relative-error summaries use positive exact counts; zero-count accuracy is reported separately.
A zero-count synopsis answer is safe only when it equals zero. R² is undefined for constant ground truth.
Skewness-Align is a ranking score, so its score is not treated as a relative-error regression prediction.
The strict oracle accepts exactly the safe queries (zero allowed violations), unlike a delta-tolerant oracle.

In [0]:
%python
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

run_dir = Path(spark.conf.get("synopsislake.e2e.runDir"))
manifest = json.loads((run_dir / "manifest.json").read_text())
eps, delta = float(manifest["epsilon"]), float(manifest["delta"])
measurements = pd.read_csv(run_dir / "measurements.csv")
queries = pd.read_csv(run_dir / "queries.csv")
setup = pd.read_csv(run_dir / "setup.csv")
for column in ["accepted", "fallback"]:
    measurements[column] = measurements[column].astype(str).str.lower().map({"true": True, "false": False})
results = measurements.merge(queries, on="qId", how="left", validate="many_to_one")
groups = ["merge_rate", "access_mode", "method"]
ok = results[results.status == "ok"].copy()
failures = results[results.status != "ok"]
# Save all attempts, including failures. CSVs remain available if Parquet export is unavailable.
results.to_parquet(run_dir / "per_query_results.parquet", index=False)

def answer_metrics(estimate, exact):
    estimate, exact = np.asarray(estimate, float), np.asarray(exact, float)
    error = estimate - exact
    positive, zero = exact > 0, exact == 0
    rel = np.abs(error[positive]) / exact[positive]
    denominator = np.sum((exact - exact.mean()) ** 2)
    return dict(mae=np.abs(error).mean(), mse=np.mean(error ** 2), rmse=np.sqrt(np.mean(error ** 2)),
                r2=1 - np.sum(error ** 2) / denominator if denominator > 0 else np.nan,
                mean_relative_error=rel.mean() if len(rel) else np.nan,
                median_relative_error=np.median(rel) if len(rel) else np.nan,
                p95_relative_error=np.quantile(rel, .95) if len(rel) else np.nan,
                positive_count_queries=int(positive.sum()), zero_count_queries=int(zero.sum()),
                zero_count_correctness=float((estimate[zero] == 0).mean()) if zero.any() else np.nan)

def is_safe(estimate, exact):
    estimate, exact = np.asarray(estimate, float), np.asarray(exact, float)
    return np.isfinite(estimate) & np.where(exact > 0, np.abs(estimate - exact) <= eps * exact, estimate == 0)

def wilson(successes, total):
    if total == 0:
        return np.nan, np.nan
    p, z = successes / total, 1.959963984540054
    den = 1 + z*z/total
    mid = (p + z*z/(2*total))/den
    half = z*np.sqrt(p*(1-p)/total + z*z/(4*total*total))/den
    return max(0., mid-half), min(1., mid+half)

def score_metrics(score, estimate, exact):
    score, estimate, exact = (np.asarray(x, float) for x in (score, estimate, exact))
    finite_score = np.isfinite(score)
    bad = ~is_safe(estimate, exact)
    labels = bad[finite_score]
    ranks = pd.Series(score[finite_score]).rank(method="average").to_numpy()
    n_bad, n_good = int(labels.sum()), int((~labels).sum())
    auc = ((ranks[labels].sum() - n_bad*(n_bad+1)/2)/(n_bad*n_good)
           if n_bad and n_good else np.nan)
    positive = finite_score & np.isfinite(estimate) & (exact > 0)
    errors = np.abs(estimate[positive]-exact[positive])/exact[positive]
    score_ranks, error_ranks = pd.Series(score[positive]).rank(), pd.Series(errors).rank()
    correlation = (score_ranks.corr(error_ranks) if len(errors) > 1 and
                   score_ranks.nunique() > 1 and error_ranks.nunique() > 1 else np.nan)
    return dict(unsafe_query_roc_auc=auc, spearman_score_vs_relative_error=correlation,
                finite_score_queries=int(finite_score.sum()))

# Repeated requests should give the same answers and routes.
consistency = ok.groupby(groups + ["qId"])[["answer", "synopsis_estimate", "score", "accepted", "fallback"]].nunique(dropna=False)
assert (consistency <= 1).all().all(), "Answers/routes changed across repetitions; investigate before reporting."
unique = ok.sort_values("repetition").drop_duplicates(groups + ["qId"])
latency_rows, accuracy_rows, gate_rows = [], [], []
for key, attempts in results.groupby(groups):
    identity = dict(zip(groups, key))
    measured = attempts[attempts.status == "ok"]
    row = dict(identity, attempts=len(attempts), failures=int((attempts.status != "ok").sum()),
               unique_queries=measured.qId.nunique())
    for name, subset in [("all", measured), ("accepted", measured[measured.accepted]), ("fallback", measured[measured.fallback])]:
        values = subset.total_ms
        row.update({f"{name}_mean_ms": values.mean(), f"{name}_median_ms": values.median(),
                    f"{name}_p95_ms": values.quantile(.95), f"{name}_requests": len(subset)})
    latency_rows.append(row)
for key, group in unique.groupby(groups):
    identity = dict(zip(groups, key))
    accuracy_rows.append(dict(identity, answer_type="returned", **answer_metrics(group.answer, group.supplied_exact)))
    if key[2] == "skewness_align":
        accuracy_rows.append(dict(identity, answer_type="synopsis_before_fallback",
                                  **answer_metrics(group.synopsis_estimate, group.supplied_exact)))
        safe = is_safe(group.synopsis_estimate, group.supplied_exact)
        accepted = group.accepted.to_numpy(bool)
        violations = int((accepted & ~safe).sum())
        lower, upper = wilson(violations, int(accepted.sum()))
        gate_rows.append(dict(identity, queries=len(group), accepted_queries=int(accepted.sum()),
            coverage=accepted.mean(), fallback_rate=(~accepted).mean(),
            accepted_violation_rate=violations/accepted.sum() if accepted.any() else np.nan,
            violation_ci95_low=lower, violation_ci95_high=upper,
            strict_oracle_coverage=safe.mean(), safe_query_acceptance=(accepted & safe).sum()/safe.sum() if safe.any() else np.nan,
            missed_safe_coverage=(~accepted & safe).mean(), unsafe_acceptance_share=(accepted & ~safe).mean(),
            returned_violation_rate=(~is_safe(group.answer, group.supplied_exact)).mean(),
            **score_metrics(group.score, group.synopsis_estimate, group.supplied_exact)))
latency = pd.DataFrame(latency_rows)
accuracy = pd.DataFrame(accuracy_rows)
gate_metrics = pd.DataFrame(gate_rows)

# Paired bootstrap resamples query IDs and retains all repetitions of each sampled query.
pair_keys = ["merge_rate", "access_mode", "repetition", "qId"]
paired = (ok.pivot(index=pair_keys, columns="method", values="total_ms")
          .reindex(columns=["always_exact", "skewness_align"]).dropna().reset_index())
comparisons = []
for key, group in paired.groupby(["merge_rate", "access_mode"]):
    identity = dict(zip(["merge_rate", "access_mode"], key))
    complete = (group.groupby("qId").size() == int(manifest["repetitions"]))
    group = group[group.qId.isin(complete[complete].index)]
    if group.empty:
        continue
    means = group.groupby("qId")[["always_exact", "skewness_align"]].mean().to_numpy()
    rng = np.random.default_rng(int(manifest["seed"]))
    boot_speed, boot_difference = [], []
    for _ in range(1000):
        sample = means[rng.integers(0, len(means), len(means))]
        boot_speed.append(sample[:, 0].mean()/sample[:, 1].mean())
        boot_difference.append((sample[:, 0] - sample[:, 1]).mean())
    same_config = results[(results.merge_rate == key[0]) & (results.access_mode == key[1])]
    comparisons.append(dict(identity, complete_pairs=len(group), complete_queries=len(means),
        valid_no_failures=bool((same_config.status == "ok").all()),
        mean_latency_speedup=group.always_exact.mean()/group.skewness_align.mean(),
        speedup_ci95_low=np.quantile(boot_speed, .025), speedup_ci95_high=np.quantile(boot_speed, .975),
        mean_ms_saved=(group.always_exact-group.skewness_align).mean(),
        saved_ms_ci95_low=np.quantile(boot_difference, .025), saved_ms_ci95_high=np.quantile(boot_difference, .975),
        p95_latency_ratio=group.always_exact.quantile(.95)/group.skewness_align.quantile(.95),
        fraction_pairs_faster=(group.skewness_align < group.always_exact).mean()))
comparison = pd.DataFrame(comparisons)
components = ok.groupby(groups)[["synopsis_load_ms", "beta_ms", "score_and_answer_ms", "routing_ms", "exact_ms", "total_ms"]].mean().reset_index()
components["other_ms"] = components.total_ms - components[["synopsis_load_ms", "beta_ms", "score_and_answer_ms", "routing_ms", "exact_ms"]].sum(axis=1)
for name, table in [("latency", latency), ("accuracy", accuracy), ("gate_metrics", gate_metrics), ("comparison", comparison), ("components", components)]:
    table.to_csv(run_dir / f"{name}.csv", index=False)
    print(name)
    display(table)
print("Offline and initialization costs (not query latency):")
display(setup)
if not failures.empty:
    print("INVALID RUN: failed requests are retained. Resolve them before claiming a performance or accuracy benefit.")
    display(failures[["qId", "method", "merge_rate", "access_mode", "error"]])

# Diagnostic groups are computed after timing; they are never used to gate test queries.
diagnostics = unique[unique.method == "skewness_align"].copy()
diagnostics["safe"] = is_safe(diagnostics.synopsis_estimate, diagnostics.supplied_exact)
diagnostics["safe_but_rejected"] = diagnostics.safe & ~diagnostics.accepted
diagnostics["area"] = (diagnostics.maxLon-diagnostics.minLon)*(diagnostics.maxLat-diagnostics.minLat)
diagnostics["selectivity"] = diagnostics.supplied_exact / int(manifest["raw_count"])
diagnostics["count_bucket"] = pd.cut(diagnostics.supplied_exact, [-1, 0, 9, 99, 999, np.inf], labels=["0", "1-9", "10-99", "100-999", "1000+"])
diagnostics["cells_bucket"] = pd.cut(diagnostics.cells_touched, [-1, 0, 4, 16, 64, np.inf], labels=["0", "1-4", "5-16", "17-64", "65+"])
rejection_rows = []
for dimension in ["count_bucket", "cells_bucket"]:
    for key, group in diagnostics.groupby(["merge_rate", "access_mode", dimension], observed=True):
        rejection_rows.append(dict(merge_rate=key[0], access_mode=key[1], dimension=dimension, bucket=str(key[2]),
            queries=len(group), safe_queries=int(group.safe.sum()), safe_but_rejected=int(group.safe_but_rejected.sum()),
            safe_rejection_rate=group.safe_but_rejected.sum()/group.safe.sum() if group.safe.any() else np.nan))
rejection_summary = pd.DataFrame(rejection_rows)
rejection_summary.to_csv(run_dir / "safe_rejection_summary.csv", index=False)
diagnostics.to_parquet(run_dir / "query_diagnostics.parquet", index=False)
display(rejection_summary)

if not ok.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for key, group in ok.groupby(groups):
        values = np.sort(group.total_ms.to_numpy())
        axes[0].plot(values, np.arange(1, len(values)+1)/len(values), label=f"{key[2]} r={key[0]:.3f} {key[1]}")
    axes[0].set(xlabel="Request latency (ms)", ylabel="Cumulative fraction", xscale="log")
    axes[0].legend(fontsize=7)
    plot = components.copy()
    plot.index = [f"{r.method}\nr={r.merge_rate:.3f} {r.access_mode}" for r in plot.itertuples()]
    plot[["synopsis_load_ms", "beta_ms", "score_and_answer_ms", "routing_ms", "exact_ms", "other_ms"]].plot.bar(stacked=True, ax=axes[1])
    axes[1].set(ylabel="Mean request latency (ms)", xlabel="")
    axes[1].tick_params(axis="x", rotation=25)
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(run_dir / "latency_and_components.png", dpi=160)
    plt.show()
report = ["# Skewness-Align end-to-end experiment", f"Run: {manifest['run_id']}",
          f"Raw representation: {manifest['raw_format']}; cache policy: {manifest['raw_cache_policy']}",
          f"Tolerance epsilon={eps}; calibration target delta={delta}; failures={len(failures)}.",
          "Pilot results apply to the sampled workload and recorded cluster. Reload is not a cold-storage guarantee.",
          "Approximation uses the original Java implementation; exact fallback and always-exact both execute Spark COUNT.",
          "Score and synopsis-answer cost are combined. No learned model inference is present.",
          "Accuracy is per unique query; timing repeats are clustered by query for the speedup confidence interval."]
for name, table in [("Comparison", comparison), ("Latency", latency), ("Gate", gate_metrics), ("Accuracy", accuracy), ("Setup", setup)]:
    report += [f"\n## {name}", "```", table.to_string(index=False), "```"]
(run_dir / "report.md").write_text("\n\n".join(report))
print(f"All metrics, per-query results, plot, manifest, and report saved in:\n{run_dir}")